# Real-Time Bitcoin Price Processing with Amazon Lambda

This notebook demonstrates a working system for fetching and storing real-time Bitcoin price data.  
The system uses:
- **`utils/amazon_lambda_utils.py`** to fetch BTC price and upload to S3
- **`ingest/lambda_function.py`** to run a scheduled fetch every 15 seconds for 2 minutes

We also discuss the AWS Lambda deployment for serverless automation.


## Folder Structure
TutorTask65_Spring2025_Real_Time_Bitcoin_Price_Processing_with_Amazon_Lambda/
├── ingest/
│   └── lambda_function.py            ← Local runner (fetches BTC price & uploads to S3)
├── utils/
│   └── amazon_lambda_utils.py        ← Core logic for price fetch & S3 upload
├── notebooks/
│   └── Amazon_Lambda.API.ipynb       ← Main tutorial notebook with markdowns and execution
├── aws_lambda_package/
│   ├── lambda_function.py            ← AWS Lambda deployable script
│   ├── requirements.txt              ← Deployment dependencies
│   └── lambda_deploy_package.zip     ← Zipped package for AWS upload
├── docker_data605_style/             ← Local Docker dev environment (optional)
├── README.md                         ← Project overview and execution instructions
├── .gitignore                        ← Prevents unnecessary files from being tracked


In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from utils.amazon_lambda_utils import get_live_btc_price, upload_to_s3


Fetching live Bitcoin price...
Fetched data: {'price': 95030.145, 'timestamp': '2025-04-26T02:40:08.908719'}

 Uploading to AWS S3...
Upload successful! File uploaded to: s3://ruthvick-btc-bucket/btc_price_latest.json


## Fetch a single live Bitcoin price


In [ ]:
btc_price = get_live_btc_price()
print("BTC Price:", btc_price)


##  Upload the result to S3


In [ ]:
bucket = "ruthvick-btc-bucket"
key = "btc_data/btc_price_manual_test.json"
upload_to_s3(btc_price, bucket, key)
print(f"Uploaded to s3://{bucket}/{key}")


## Module Functionality Verification

To ensure correctness and modular integrity, the core components of the project were tested independently before integration.

## utils/amazon_lambda_utils.py

This module defines two key functions:

- `get_live_btc_price()`: Connects to the Coinbase API and retrieves the current Bitcoin price in USD with a UTC timestamp.
- `upload_to_s3(data, bucket_name, key)`: Uploads the given data to an AWS S3 bucket in JSON format using Boto3.

Both were tested separately, confirming accurate price retrieval and successful uploads to the S3 bucket.

## ingest/lambda_function.py

This script runs the system locally as a scheduled fetcher. It:

- Retrieves Bitcoin prices every 15 seconds.
- Automatically stops after 2 minutes (configurable via a variable).
- Generates unique filenames using UTC-based timestamps.
- Uploads each JSON-formatted data point to S3 at the path:

  `s3://ruthvick-btc-bucket/btc_data/btc_price_<timestamp>.json`

Tested results show:
- Stable API connectivity and formatting.
- Valid S3 authentication and uploads.
- Functional runtime limitation and loop interval.
Output (sample):

pgsql
Copy
Edit
Fetched BTC Price: {'price': 94755.36, 'timestamp': '2025-05-01T04:20:06.844046'}
Data uploaded to s3://ruthvick-btc-bucket/btc_data/btc_price_20250501T042006.json

Note: The 2-minute auto-stop behavior is currently implemented only in the local version (ingest/lambda_function.py). It has not yet been incorporated into the deployed AWS Lambda

Things Under Consideration 

WebSocket-Based Streaming
Replace the polling-based system with real-time WebSocket streaming (as originally proposed) to capture every price update rather than sampling every 15 seconds.

Price Anomaly Detection 
Add a lightweight analysis function to detect sudden spikes/drops and log or alert based on thresholds.

Email/SMS Alerts Use AWS SNS or similar service to notify users if BTC price crosses critical thresholds.

AWS Glue or Athena Integration
 Enable analysis of stored S3 data using AWS Glue ETL jobs or direct Athena SQL queries.

Time-Limited Lambda Execution in AWS 
Currently, the 2-minute loop is implemented only locally. A similar timed structure within AWS Lambda isn’t feasible in one call due to the 15-minute max execution time and statelessness. Instead, longer-duration collection would need to be handled via:

Step Functions

SQS for chaining messages

Or breaking into repeated EventBridge triggers

